# 06 · Clean and combine

**Milestone 2, item 2.1** · see `docs/MILESTONES.md`

## What we're looking for

Milestone 1 left the data in six places, on different calendars:

| Source | Frequency | Starts |
|---|---|---|
| BLS retail prices and CPI (notebook 01, raw file) | Monthly | 2015 |
| ERS price spreads | Monthly | 2015 |
| NASS herd | Once a year (January 1) | 2010 |
| NASS slaughter | Monthly | 2010 |
| Drought in cattle country | Weekly, averaged to monthly | 2010 |
| BLS grocery categories | Monthly | 2015 |

This notebook builds **one monthly table**, `data/processed/master_monthly.csv`, that the
analysis (Milestone 4) and the app (Milestone 5) both read, plus a **data dictionary**
(`docs/DATA_DICTIONARY.md`) describing every column.

Decided together (D27):
- **Range:** January 2010 to the latest month. Price columns are empty before 2015 (D3);
  herd, slaughter and drought go back to 2010 (D24).
- **Inflation-adjusted prices** are in **dollars of the latest month** (September 2026 at
  the time of writing), so every past price is shown in today's money.
- **The herd** is counted once a year, so each January 1 count is **repeated for every
  month of that year**, never estimated in between.
- **Single missing months** (October 2025 in every BLS series) are filled with the average
  of the months on either side and flagged (D18).

**Steps**
1. Setup and load everything
2. BLS prices and CPI: tidy, fill October 2025, flag it
3. Inflation-adjusted prices
4. ERS spreads, wide
5. Herd (yearly → every month of the year), slaughter, drought
6. Grocery categories: fill and widen
7. Join into one table
8. Checks
9. Save the table and write the data dictionary

## Step 1 · Setup and load everything

In [1]:
import json
from pathlib import Path

import pandas as pd
from dotenv import find_dotenv

env_path = find_dotenv()
PROJECT_ROOT = Path(env_path).parent if env_path else Path.cwd().parent
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
pd.set_option("display.max_rows", 120)
pd.set_option("display.width", 160)

bls_file = sorted(RAW_DIR.glob("bls_prices_*.json"))[-1]
bls_raw = json.loads(bls_file.read_text())

ers = pd.read_csv(PROCESSED_DIR / "ers_price_spreads_monthly.csv", parse_dates=["date"])
herd = pd.read_csv(PROCESSED_DIR / "nass_herd_jan1.csv")
slaughter = pd.read_csv(PROCESSED_DIR / "nass_slaughter_monthly.csv", parse_dates=["date"])
drought = pd.read_csv(PROCESSED_DIR / "drought_cattle_country_monthly.csv", parse_dates=["date"])
cpi_cats = pd.read_csv(PROCESSED_DIR / "cpi_grocery_categories_monthly.csv", parse_dates=["date"])

print("BLS raw file:", bls_file.name)
for name, df in [("ers", ers), ("herd", herd), ("slaughter", slaughter), ("drought", drought), ("cpi_cats", cpi_cats)]:
    print(f"{name:10} {len(df):>6,} rows")

BLS raw file: bls_prices_20261026.json
ers         2,808 rows
herd           17 rows
slaughter     201 rows
drought       202 rows
cpi_cats    2,115 rows


### How to read it
The BLS file used (the latest download from notebook 01) and the row count of each
processed file. If any file is missing, the notebook that creates it hasn't been run.

## Step 2 · BLS prices and CPI

We keep the four series chosen in D15 and give them plain column names:

| Series | Column | Units |
|---|---|---|
| APU0000703112 | `beef_price` | $ per lb, ground beef |
| APU0000FD3101 | `pork_price` | $ per lb, all pork chops |
| APU0000FF1101 | `chicken_price` | $ per lb, boneless chicken breast |
| CUUR0000SA0 | `cpi_all_items` | index, 1982–84 = 100 |

### Filling single missing months (D18)

The function below fills a gap **only** when it's one or two months long and has real
values on both sides; longer gaps stay empty. It uses straight-line interpolation, which
for a single missing month is exactly the average of the month before and after. It also
records which columns it filled in each row, in a `filled_columns` column, so a filled
value can never be mistaken for a published one.

In [2]:
CHOSEN = {"APU0000703112": "beef_price", "APU0000FD3101": "pork_price",
          "APU0000FF1101": "chicken_price", "CUUR0000SA0": "cpi_all_items"}

rows = []
for s in bls_raw["Results"]["series"]:
    if s["seriesID"] in CHOSEN:
        for o in s["data"]:
            if o["period"].startswith("M") and o["period"] != "M13":
                rows.append({"date": pd.Timestamp(int(o["year"]), int(o["period"][1:]), 1),
                             "column": CHOSEN[s["seriesID"]],
                             "value": pd.to_numeric(o["value"], errors="coerce")})
bls = pd.DataFrame(rows).pivot_table(index="date", columns="column", values="value")
bls = bls.reindex(pd.date_range(bls.index.min(), bls.index.max(), freq="MS"))[list(CHOSEN.values())]

FILL_LOG = []   # (date, column) pairs that were filled

def fill_short_gaps(df, max_gap=2):
    """Fill gaps of up to max_gap months that have values on both sides; log what was filled."""
    out = df.copy()
    for col in df.columns:
        s = df[col]
        is_na = s.isna()
        run_id = (is_na != is_na.shift()).cumsum()
        for _, run in s[is_na].groupby(run_id[is_na]):
            start, end = run.index.min(), run.index.max()
            inside = s.loc[:start].notna().any() and s.loc[end:].notna().any()
            if len(run) <= max_gap and inside:
                out[col] = out[col].where(~out.index.isin(run.index), s.interpolate(limit_area="inside"))
                FILL_LOG.extend((d, col) for d in run.index)
    return out

bls = fill_short_gaps(bls)
print("Filled:", sorted({(d.strftime("%Y-%m"), col) for d, col in FILL_LOG}))
bls.loc["2025-08-01":"2025-12-01"].round(3)

Filled: [('2025-10', 'beef_price'), ('2025-10', 'chicken_price'), ('2025-10', 'cpi_all_items'), ('2025-10', 'pork_price')]


column,beef_price,pork_price,chicken_price,cpi_all_items
2025-08-01,6.318,4.299,4.212,323.976
2025-09-01,6.323,4.349,4.172,324.800
2025-10-01,6.432,4.223,4.161,324.461
2025-11-01,6.540,4.097,4.150,324.122
2025-12-01,6.687,4.298,4.153,324.054


### How to read it
- **Filled** lists every (month, column) the function filled. We expect October 2025 for
  all four columns and nothing else.
- The table shows August to December 2025, so you can see each filled October value sits
  halfway between September and November.

### What it means
Only **October 2025** was filled, in all four columns, as expected. Each filled value is
halfway between its neighbors: ground beef, for example, is $6.432, the average of
$6.323 (September) and $6.540 (November). No other month in the BLS data needed filling.

## Step 3 · Inflation-adjusted prices (D5, D27)

To show a past price in today's money, multiply it by how much prices in general have
risen since then:

**price in today's dollars = price then × (CPI today ÷ CPI then)**

"Today" is the **latest month** in the data. Example: if the CPI is 336.9 today and was
237.0 in a month of 2015, a $4.20 price that month is worth 4.20 × 336.9 ÷ 237.0 =
**$5.97 in today's dollars**. If beef costs more than that now, it rose faster than
inflation.

The new columns end in `_real` (economists' word for inflation-adjusted, as opposed to
*nominal*, the price at the time).

In [3]:
BASE_MONTH = bls["cpi_all_items"].last_valid_index()
base_cpi = bls.loc[BASE_MONTH, "cpi_all_items"]

for meat in ["beef", "pork", "chicken"]:
    bls[f"{meat}_price_real"] = bls[f"{meat}_price"] * base_cpi / bls["cpi_all_items"]

print(f"Real prices are in {BASE_MONTH:%B %Y} dollars (CPI {base_cpi:.3f})")
summary = pd.DataFrame({
    "2015 avg, then": bls.loc["2015", ["beef_price", "pork_price", "chicken_price"]].mean().values,
    "2015 avg, today's $": bls.loc["2015", ["beef_price_real", "pork_price_real", "chicken_price_real"]].mean().values,
    "latest": bls.loc[BASE_MONTH, ["beef_price", "pork_price", "chicken_price"]].values,
}, index=["beef", "pork", "chicken"])
summary["real change %"] = (summary["latest"] / summary["2015 avg, today's $"] - 1) * 100
summary.round(2)

Real prices are in September 2026 dollars (CPI 336.940)


,"2015 avg, then","2015 avg, today's $",latest,real change %
beef,4.17,5.92,6.79,14.65
pork,3.86,5.49,4.22,-23.22
chicken,3.41,4.85,4.20,-13.31


### How to read it
For each meat: the 2015 average price as it was then, the same price in today's dollars,
the latest price, and the **real change**, how much the price changed *after* removing
general inflation.

### What it means
Measured in September 2026 dollars:

| | 2015 average, in today's dollars | September 2026 | Real change |
|---|---|---|---|
| Ground beef | $5.92 | $6.79 | **+15%** |
| Pork chops | $5.49 | $4.22 | **−23%** |
| Chicken breast | $4.85 | $4.20 | **−13%** |

- **Beef is the only meat that got more expensive in real terms**: about 15% above its
  2015 price even after allowing for inflation.
- **Pork and chicken got cheaper in real terms.** Their prices rose less than everything
  else, so compared with what people earn and spend overall, they cost less than in 2015.

This is the same conclusion as notebook 01's rough comparison, now measured properly month
by month, and it's what the app's "Adjusted for inflation" toggle will show.

## Step 4 · ERS price spreads, wide

The ERS table from notebook 02 is long (one row per month, meat and measure). Here it
becomes one column per meat and measure, named `ers_<meat>_<measure>`, for example
`ers_beef_farmer_share_pct`. Values stay in ERS's units: cents per pound of retail
equivalent, or percent of the retail value. Chicken's January–March 2026 gap stays empty
(D22): it's three months long, beyond the fill rule.

In [4]:
ers_wide = ers.assign(column="ers_" + ers["meat"].str.lower() + "_" + ers["measure"]) \
             .pivot_table(index="date", columns="column", values="value")
print(ers_wide.shape[1], "ERS columns")
list(ers_wide.columns)

20 ERS columns


['ers_beef_farmer_share_pct',
 'ers_beef_net_farm',
 'ers_beef_packer_share_pct',
 'ers_beef_retail',
 'ers_beef_spread_farm_to_wholesale',
 'ers_beef_spread_wholesale_to_retail',
 'ers_beef_store_share_pct',
 'ers_beef_wholesale',
 'ers_chicken_retail',
 'ers_chicken_spread_wholesale_to_retail',
 'ers_chicken_store_share_pct',
 'ers_chicken_wholesale',
 'ers_pork_farmer_share_pct',
 'ers_pork_net_farm',
 'ers_pork_packer_share_pct',
 'ers_pork_retail',
 'ers_pork_spread_farm_to_wholesale',
 'ers_pork_spread_wholesale_to_retail',
 'ers_pork_store_share_pct',
 'ers_pork_wholesale']

## Step 5 · Herd, slaughter and drought

- **Herd:** each January 1 count is repeated for every month of that year (D27). A column
  named `herd_…_jan1` makes clear it's the start-of-year count, not a monthly count.
- **Slaughter** and **drought** are already monthly; they get `slaughter_` and `drought_`
  prefixes so every column says where it came from.

In [5]:
months = pd.date_range("2010-01-01", BASE_MONTH, freq="MS")

herd_monthly = pd.DataFrame(index=months)
herd_monthly["year"] = herd_monthly.index.year
herd_monthly = herd_monthly.merge(herd.drop(columns="reference_date"), on="year", how="left").set_index(months)
herd_monthly = herd_monthly.drop(columns="year").add_prefix("herd_").add_suffix("_jan1")

slaughter_m = slaughter.set_index("date").add_prefix("slaughter_")
drought_m = drought.set_index("date").add_prefix("drought_")

herd_monthly.loc["2025-11-01":"2026-02-01"]

,herd_all_cattle_jan1,herd_beef_cows_jan1,herd_beef_replacement_heifers_jan1
2025-11-01,86472200.0,27892000.0,4672500.0
2025-12-01,86472200.0,27892000.0,4672500.0
2026-01-01,86155300.0,27607200.0,4714200.0
2026-02-01,86155300.0,27607200.0,4714200.0


### How to read it
Four months either side of a new year: the herd values stay the same through December
2025 and change in January 2026, when the next count applies.

## Step 6 · Grocery categories

The 15 CPI categories from notebook 05 get the same October 2025 fill, then one column
each holding the **rebased index** (2015 average = 100, see "Key terms: price indexes and
rebasing" in DATA_SOURCES.md), named `cpi_<category>_2015eq100`.

The rebased index is recalculated after filling, so a filled October doesn't shift the
2015 average (it doesn't, since 2015 has no gaps, but recalculating keeps it simple and
correct).

In [6]:
import re

def slug(text):
    return re.sub(r"[^a-z0-9]+", "_", text.lower()).strip("_")

cats = cpi_cats.pivot_table(index="date", columns="category", values="value")
cats = cats.reindex(pd.date_range(cats.index.min(), cats.index.max(), freq="MS"))
cats = fill_short_gaps(cats)
cats_idx = cats / cats.loc["2015"].mean() * 100
cats_idx.columns = ["cpi_" + slug(col) + "_2015eq100" for col in cats_idx.columns]
print(len(cats_idx.columns), "category columns")
list(cats_idx.columns)

15 category columns


['cpi_all_items_2015eq100',
 'cpi_beef_and_veal_2015eq100',
 'cpi_cereals_and_bakery_2015eq100',
 'cpi_dairy_2015eq100',
 'cpi_eggs_2015eq100',
 'cpi_food_at_home_2015eq100',
 'cpi_food_away_from_home_2015eq100',
 'cpi_fresh_fruits_and_vegetables_2015eq100',
 'cpi_fruits_and_vegetables_2015eq100',
 'cpi_meats_poultry_fish_and_eggs_2015eq100',
 'cpi_nonalcoholic_beverages_2015eq100',
 'cpi_other_food_at_home_packaged_2015eq100',
 'cpi_pork_2015eq100',
 'cpi_poultry_2015eq100',
 'cpi_processed_fruits_and_vegetables_2015eq100']

## Step 7 · Join into one table

Everything is joined on the month, over the full range January 2010 to the latest month.
The `filled_columns` column lists, for each month, any columns that were filled (empty
for most months).

In [7]:
master = pd.DataFrame(index=months)
for part in [bls, ers_wide, herd_monthly, slaughter_m, drought_m, cats_idx]:
    master = master.join(part, how="left")

fills = pd.DataFrame(FILL_LOG, columns=["date", "column"])
fills["column"] = fills["column"].map(lambda col: "cpi_" + slug(col) + "_2015eq100" if col in cats.columns else col)
filled_text = fills.groupby("date")["column"].apply(lambda s: ";".join(sorted(set(s))))
master["filled_columns"] = filled_text.reindex(master.index).fillna("")
master.index.name = "date"

print(f"{master.shape[0]} months x {master.shape[1]} columns, "
      f"{master.index.min():%Y-%m} to {master.index.max():%Y-%m}")

201 months x 55 columns, 2010-01 to 2026-09


## Step 8 · Checks

**What we're looking for:**
1. One row per month, no duplicates, no missing months.
2. For each column, the first and last month with data, and how many months inside that
   range are empty. Expected empty stretches: price, ERS and grocery columns before 2015
   (they start in 2015), and chicken's ERS columns for January–March 2026. Nothing else.
3. The filled months are exactly what Step 2 reported.

In [8]:
assert master.index.is_unique and len(master) == len(months), "row problem"

col_check = []
for col in master.columns.drop("filled_columns"):
    s = master[col]
    first, last = s.first_valid_index(), s.last_valid_index()
    inner_gaps = s.loc[first:last].isna().sum() if first is not None else None
    col_check.append({"column": col, "first": first.strftime("%Y-%m") if first is not None else "none",
                      "last": last.strftime("%Y-%m") if last is not None else "none",
                      "empty_months_inside": inner_gaps})
col_check = pd.DataFrame(col_check)

print("Columns with gaps inside their range:")
print(col_check[col_check["empty_months_inside"] > 0].to_string(index=False) or "none")
print()
print("Months with filled values:", master.loc[master["filled_columns"] != ""].index.strftime("%Y-%m").tolist())
col_check

Columns with gaps inside their range:
                                column   first    last  empty_months_inside
                    ers_chicken_retail 2015-01 2026-09                    3
ers_chicken_spread_wholesale_to_retail 2015-01 2026-09                    3
           ers_chicken_store_share_pct 2015-01 2026-09                    3
                 ers_chicken_wholesale 2015-01 2026-09                    3

Months with filled values: ['2025-10']


,column,first,last,empty_months_inside
0,beef_price,2015-01,2026-09,0
1,pork_price,2015-01,2026-09,0
2,chicken_price,2015-01,2026-09,0
3,cpi_all_items,2015-01,2026-09,0
4,beef_price_real,2015-01,2026-09,0
5,pork_price_real,2015-01,2026-09,0
6,chicken_price_real,2015-01,2026-09,0
7,ers_beef_farmer_share_pct,2015-01,2026-09,0
8,ers_beef_net_farm,2015-01,2026-09,0
9,ers_beef_packer_share_pct,2015-01,2026-09,0


### How to read it
- **Columns with gaps inside their range** should list only the chicken ERS columns,
  each with 3 empty months (January–March 2026).
- **Months with filled values** should be October 2025 only.
- The full table lists every column with its first and last month.

### What it means
Everything checks out:
- **201 months** (January 2010 to September 2026), one row each.
- The only columns with empty months inside their range are the **four chicken ERS
  columns**, each missing January–March 2026, the known gap (D22).
- The only filled month is **October 2025**.
- Price, ERS and grocery columns start in January 2015; herd, slaughter and drought start
  in January 2010, as decided (D27).

## Step 9 · Save the table and write the data dictionary

The table is saved as `data/processed/master_monthly.csv` (committed, D9, D12).

The **data dictionary** is generated from the descriptions below and written to
`docs/DATA_DICTIONARY.md`, so it always matches the columns actually in the table. If a
column has no description, the notebook stops with an error instead of writing an
incomplete dictionary.

In [9]:
MEAT_PRICE = {"beef": "ground beef, 100% beef (BLS APU0000703112)",
              "pork": "all pork chops (BLS APU0000FD3101)",
              "chicken": "boneless chicken breast (BLS APU0000FF1101)"}
ERS_MEASURE = {
    "retail": ("Retail value: what shoppers pay", "cents per lb, retail equivalent"),
    "wholesale": ("Wholesale value: what the packer/processor gets", "cents per lb, retail equivalent"),
    "net_farm": ("Net farm value: what the rancher/farmer gets, minus byproducts", "cents per lb, retail equivalent"),
    "spread_farm_to_wholesale": ("Farm-to-wholesale spread (roughly the packer's part)", "cents per lb, retail equivalent"),
    "spread_wholesale_to_retail": ("Wholesale-to-retail spread (roughly the store's part)", "cents per lb, retail equivalent"),
    "farmer_share_pct": ("Farm share of the retail value", "percent"),
    "packer_share_pct": ("Packer share of the retail value", "percent"),
    "store_share_pct": ("Store share of the retail value", "percent"),
}

DESCRIBE = {"cpi_all_items": ("CPI, all items, U.S. city average, not seasonally adjusted (CUUR0000SA0)",
                              "index, 1982-84 = 100", "BLS"),
            "filled_columns": ("Columns whose value this month was filled (D18); empty if none",
                               "text, semicolon-separated", "notebook 06")}
for meat, label in MEAT_PRICE.items():
    DESCRIBE[f"{meat}_price"] = (f"Average U.S. retail price, {label}", "$ per lb, at the time", "BLS")
    DESCRIBE[f"{meat}_price_real"] = (f"Same price in {BASE_MONTH:%B %Y} dollars (adjusted with CPI, all items)",
                                      f"$ per lb, {BASE_MONTH:%b %Y} dollars", "calculated")
for col in ers_wide.columns:
    meat, measure = col.split("_", 2)[1], col.split("_", 2)[2]
    text, units = ERS_MEASURE[measure]
    DESCRIBE[col] = (f"{meat.capitalize()}: {text}", units, "USDA ERS (spreads and shares calculated, D22)")
HERD_TEXT = {"all_cattle": "All cattle and calves", "beef_cows": "Beef cows",
             "beef_replacement_heifers": "Beef replacement heifers, 500 lb and over"}
for key, text in HERD_TEXT.items():
    DESCRIBE[f"herd_{key}_jan1"] = (f"{text} on January 1 of this month's year (repeated each month, D27)",
                                    "head", "USDA NASS")
DESCRIBE.update({
    "slaughter_beef_cows_slaughtered": ("Beef cows slaughtered this month, federally inspected", "head", "USDA NASS"),
    "slaughter_heifers_slaughtered": ("Heifers slaughtered this month, federally inspected", "head", "USDA NASS"),
    "slaughter_steers_slaughtered": ("Steers slaughtered this month, federally inspected", "head", "USDA NASS"),
    "slaughter_heifer_share_pct": ("Heifers as % of heifer + steer slaughter", "percent", "calculated"),
    "slaughter_beef_cows_slaughtered_12mo": ("Beef cows slaughtered in the 12 months ending this month", "head", "calculated"),
    "drought_cattle_d2_plus_pct": ("Cattle country (beef-cow-weighted states, D26): % of area in severe drought or worse", "percent of area", "U.S. Drought Monitor"),
    "drought_cattle_dsci": ("Cattle country: Drought Severity and Coverage Index", "index, 0-500", "U.S. Drought Monitor"),
    "drought_conus_d2_plus_pct": ("Lower 48 states: % of area in severe drought or worse", "percent of area", "U.S. Drought Monitor"),
    "drought_conus_dsci": ("Lower 48 states: Drought Severity and Coverage Index", "index, 0-500", "U.S. Drought Monitor"),
})
for cat in cats.columns:
    DESCRIBE["cpi_" + slug(cat) + "_2015eq100"] = (f"CPI, {cat}, rebased so the 2015 average = 100",
                                                   "index, 2015 average = 100", "BLS (rebased)")

missing_desc = [col for col in master.columns if col not in DESCRIBE]
assert not missing_desc, f"No description for: {missing_desc}"

out_path = PROCESSED_DIR / "master_monthly.csv"
master.reset_index().to_csv(out_path, index=False, float_format="%.4f")

lines = ["# Data dictionary: `data/processed/master_monthly.csv`", "",
         "Generated by `notebooks/06_clean_and_combine.ipynb`; do not edit by hand.", "",
         f"One row per month, {master.index.min():%B %Y} to {master.index.max():%B %Y} "
         f"({len(master)} rows, {master.shape[1] + 1} columns including `date`).", "",
         "| Column | Description | Units | Source | First month | Last month |",
         "|---|---|---|---|---|---|",
         "| `date` | First day of the month | date | | | |"]
checks = col_check.set_index("column")
for col in master.columns:
    text, units, source = DESCRIBE[col]
    first = checks.loc[col, "first"] if col in checks.index else ""
    last = checks.loc[col, "last"] if col in checks.index else ""
    lines.append(f"| `{col}` | {text} | {units} | {source} | {first} | {last} |")
dict_path = PROJECT_ROOT / "docs" / "DATA_DICTIONARY.md"
dict_path.write_text("\n".join(lines) + "\n", encoding="utf-8")

print("Saved:", out_path.relative_to(PROJECT_ROOT))
print("Wrote:", dict_path.relative_to(PROJECT_ROOT), f"({len(master.columns) + 1} columns described)")

Saved: data\processed\master_monthly.csv
Wrote: docs\DATA_DICTIONARY.md (56 columns described)


## What's next and why

**Item 2.2: exploratory charts** (`07_explore.ipynb`). With everything in one table, we
can look at all the series together for the first time, spot anything odd before the
formal analysis, and decide which relationships Milestone 4 should test first.